# How we decode SMS message text, and proof it works

**What this notebook is for:** our pipeline receives SMS messages as raw hex bytes (not readable text), and has to figure out how to turn those bytes into the actual message someone typed. This notebook:

1. **Shows the actual code that does this** — copied directly into the cells below, not imported from anywhere else. You can read this notebook top to bottom and see everything, with no other files to go look at.
2. **Runs that code against real messages** from our data and checks the results make sense.
3. **Documents a real bug we found and fixed** while checking this (some binary/non-text messages were being decoded into confident-looking gibberish instead of being correctly flagged as "not text").

**The short version of how decoding works:**
- Every message arrives as a hex string, e.g. `"48656c6c6f"`. Step 1 is always `bytes.fromhex(...)` to get the raw bytes back.
- Sometimes those bytes start with a small "header" (called a UDH) that isn't part of the message — it's metadata like "this message is part 2 of 3" or "this is meant for application X, not a person." We detect and remove that header before decoding.
- Each message also carries a small number called `dcs` (Data Coding Scheme) that's supposed to say which text encoding to use — plain English text (GSM-7), a stricter ASCII, Latin-1, or Unicode (UTF-16, for emoji/non-English text). In practice this number doesn't always tell the truth, and (surprisingly) it doesn't even mean the same thing between our two data sources — so we double-check by trying multiple decodings and keeping whichever one produces the most legible-looking result.

The rest of this notebook builds that up piece by piece, then proves it on real data.

In [1]:
import re
from pathlib import Path
from typing import NamedTuple

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pd.set_option("display.max_colwidth", 80)
print("repo root (used below only to find sample data files):", REPO_ROOT)

repo root (used below only to find sample data files): C:\Users\IshitaGodani\Documents\projects\spam-detection-prototype


## Step 1 — decoding GSM-7 (the classic "plain SMS text" encoding)

GSM-7 is the original SMS text encoding: it only needs 7 bits per character (not the usual 8), so phones historically "packed" 8 characters into 7 bytes to save space. To read it back, you have to unpack those bits first, then look each 7-bit value up in a character table.

`_GSM7_BASIC`/`_GSM7_EXT` below are that lookup table, straight from the GSM 03.38 telecom standard. `_unpack_septets` undoes the bit-packing. `decode_gsm7_packed` does both steps.

*(Copied as-is from `ingestion/dcs_codecs.py`.)*

In [2]:
# GSM 03.38 default alphabet, basic (single-septet) table. Codes with no
# defined character (reserved) are simply absent - _decode_gsm7_codes()
# renders those as a placeholder so they get penalized by _printable_score
# instead of silently faking a character.
_GSM7_BASIC = {
    0x00: '@', 0x01: '£', 0x02: '$', 0x03: '¥', 0x04: 'è', 0x05: 'é',
    0x06: 'ù', 0x07: 'ì', 0x08: 'ò', 0x09: 'Ç', 0x0A: '\n', 0x0B: 'Ø',
    0x0C: 'ø', 0x0D: '\r', 0x0E: 'Å', 0x0F: 'å',
    0x10: 'Δ', 0x11: '_', 0x12: 'Φ', 0x13: 'Γ', 0x14: 'Λ', 0x15: 'Ω',
    0x16: 'Π', 0x17: 'Ψ', 0x18: 'Σ', 0x19: 'Θ', 0x1A: 'Ξ',
    # 0x1B is the escape-to-extension-table code, handled separately.
    0x1C: 'Æ', 0x1D: 'æ', 0x1E: 'ß', 0x1F: 'É',
    0x20: ' ', 0x21: '!', 0x22: '"', 0x23: '#', 0x24: '¤', 0x25: '%',
    0x26: '&', 0x27: "'", 0x28: '(', 0x29: ')', 0x2A: '*', 0x2B: '+',
    0x2C: ',', 0x2D: '-', 0x2E: '.', 0x2F: '/',
    0x3A: ':', 0x3B: ';', 0x3C: '<', 0x3D: '=', 0x3E: '>', 0x3F: '?',
    0x40: '¡',
    0x5B: 'Ä', 0x5C: 'Ö', 0x5D: 'Ñ', 0x5E: 'Ü', 0x5F: '§',
    0x60: '¿',
    0x7B: 'ä', 0x7C: 'ö', 0x7D: 'ñ', 0x7E: 'ü', 0x7F: 'à',
}
for _c in range(0x30, 0x3A):  # digits 0-9
    _GSM7_BASIC[_c] = chr(_c)
for _c in range(0x41, 0x5B):  # A-Z
    _GSM7_BASIC[_c] = chr(_c)
for _c in range(0x61, 0x7B):  # a-z
    _GSM7_BASIC[_c] = chr(_c)

# GSM 03.38 extension table (reachable via escape code 0x1B) - only the
# common subset (page break, currency, brackets, common punctuation). An
# escape byte not followed by one of these is rendered as a placeholder,
# same as any other undefined code.
_GSM7_EXT = {
    0x0A: '\x0c', 0x14: '^', 0x28: '{', 0x29: '}', 0x2F: '\\', 0x3C: '[',
    0x3D: '~', 0x3E: ']', 0x40: '|', 0x65: '€',
}

_UNMAPPED = '�'  # placeholder for any code with no defined character -
                      # deliberately excluded from the "printable" count in
                      # _printable_score so a bad decode can't score well
                      # just because U+FFFD itself is technically printable.


def _unpack_septets(payload: bytes) -> list[int]:
    """LSB-first bit unpacking of GSM 7-bit-packed octets into septet
    values. Trailing bits that don't fill a full septet are padding,
    discarded (not a character)."""
    bits = ''.join(f'{b:08b}'[::-1] for b in payload)
    n = len(bits) - len(bits) % 7
    return [int(bits[i:i + 7][::-1], 2) for i in range(0, n, 7)]


def _decode_gsm7_codes(codes: list[int]) -> str:
    chars = []
    i = 0
    while i < len(codes):
        c = codes[i]
        if c == 0x1B and i + 1 < len(codes):  # extension escape
            chars.append(_GSM7_EXT.get(codes[i + 1], _UNMAPPED))
            i += 2
        else:
            chars.append(_GSM7_BASIC.get(c, _UNMAPPED))
            i += 1
    return ''.join(chars)


def decode_gsm7_packed(payload: bytes) -> str | None:
    """True GSM 7-bit packed-septet decode - one of our two data sources
    (SS7) stores GSM-7 content this way. The other source (SMPP) does NOT -
    see the plain-language note further down about why the same DCS number
    means something different per source."""
    if not payload:
        return None
    return _decode_gsm7_codes(_unpack_septets(payload))


print("GSM-7 decoder ready. Quick check - 'Hello' encoded as packed GSM-7 septets:")
print(decode_gsm7_packed(bytes.fromhex("c8329bfd6650")))

GSM-7 decoder ready. Quick check - 'Hello' encoded as packed GSM-7 septets:
Helloø


## Step 2 — the three other, simpler decoders

Not every message is GSM-7. Some are plain ASCII, some are Latin-1 (one byte = one character, no bit-packing), and some are UTF-16 (used for emoji and non-English languages like Chinese or Arabic — needs 2 bytes per character). These are all standard Python encodings, so each function below is a thin, safe wrapper: try the decode, and return `None` instead of crashing if the bytes don't fit that encoding.

*(Copied as-is from `ingestion/dcs_codecs.py`.)*

In [3]:
def decode_ascii(payload: bytes) -> str | None:
    """Strict - any byte outside 7-bit ASCII fails the whole decode, rather
    than silently accepting it the way latin-1 would."""
    if not payload:
        return None
    try:
        return payload.decode("ascii")
    except UnicodeDecodeError:
        return None


def decode_latin1(payload: bytes) -> str | None:
    """1 byte = 1 char, never raises. This is also our SMPP source's actual
    decode for GSM-7-tagged content (see the note further down)."""
    if not payload:
        return None
    return payload.decode("latin-1")


def decode_utf16be(payload: bytes) -> str | None:
    if not payload or len(payload) % 2 != 0:
        return None
    try:
        return payload.decode("utf-16-be")
    except UnicodeDecodeError:
        return None


print("3 more decoders ready.")

3 more decoders ready.


## Step 3 — cleanup, and a way to score "does this look like real text?"

Two small helpers:
- `_sanitize_for_storage` — a decoded message can legitimately contain a line break. We turn those into a plain space before saving, purely to avoid a CSV file quirk (a real multi-line message otherwise broke a later step reading a large file back in). It does NOT affect whether the decode itself was correct.
- `_printable_score` — given some decoded text, what fraction of it is made of normal, printable characters? A correct decode of real text should score close to 1.0. A wrong decode (using the wrong codec on the wrong bytes) usually produces a lot of `�` "I don't know this character" placeholders and scores low. **Important caveat we discovered:** this score is not foolproof — see the real bug further down where it was fooled.

*(Copied as-is from `ingestion/dcs_codecs.py`.)*

In [4]:
import unicodedata

# Unicode general categories treated as "good" alongside str.isprintable()
# even though isprintable() itself says no - both are legitimate in real
# UTF-16 SMS content, not decode noise: Cf (format - zero-width joiner,
# left/right-to-left marks; real emoji ZWJ sequences and bidi-marked text
# use these) and Zs (space separator - real messages have been seen using
# non-ASCII spacing, e.g. U+2000 EN QUAD, between words). Found via this
# notebook's own full-SS7-corpus run: without this, _printable_score
# flagged multiple CORRECTLY decoded UTF-16 messages (an emoji ZWJ
# sequence - "woman technologist", U+1F469 U+1F3FB U+200D U+1F4BB, a real
# well-known emoji, not corrupted; a "Maybank Alert..." message using EN
# QUAD spacing) as low-scoring/suspicious, even though nothing about the
# decode was wrong - only Cf/Zs specifically; other isprintable()=False
# categories (Cc control, Cs surrogate, Co private-use, Cn unassigned,
# Zl/Zp line/paragraph separators) are NOT added here - those really do
# only show up from wrong-codec noise, not real message content.
_ALSO_GOOD_CATEGORIES = {"Cf", "Zs"}


def _sanitize_for_storage(text: str | None) -> str | None:
    """Collapse embedded line breaks into a single space before storing."""
    if text is None:
        return None
    return re.sub(r"\r\n|\r|\n", " ", text)


def _printable_score(text: str | None) -> float:
    """Fraction of characters that are printable (or one of
    _ALSO_GOOD_CATEGORIES - see its comment above), excluding the 'unknown
    character' placeholder so a bad decode full of undefined codes can't
    score well just because the placeholder itself is technically
    printable."""
    if not text:
        return 0.0
    good = sum(
        1 for ch in text
        if ch != _UNMAPPED and (ch.isprintable() or unicodedata.category(ch) in _ALSO_GOOD_CATEGORIES)
    )
    return good / len(text)


print("printable_score('Hello world') =", _printable_score("Hello world"))
print("printable_score(replacement chars) =", _printable_score("�����"))

printable_score('Hello world') = 1.0
printable_score(replacement chars) = 0.0


## Step 4 — deciding WHICH decoder to use

Every message carries a `dcs` number that's supposed to say which decoder applies. Two complications, both found by checking real messages byte-by-byte (not assumed):

1. **The same `dcs` number means something different depending on which of our two data sources (SMPP vs SS7) the message came from.** Both sources tag "plain SMS text" content the same way, but SMPP stores it as one-byte-per-character (`decode_latin1`), while SS7 stores it as truly bit-packed GSM-7 (`decode_gsm7_packed`). Using the wrong one produces garbage. So each source below gets its OWN lookup table.
2. **A couple of `dcs` numbers are ambiguous or simply not documented for a source** — the spec lists them as meaning two different things, or doesn't say. For those, we try every plausible decoder and keep whichever one scores highest with `_printable_score` from Step 3 — i.e. whichever produces the most legible-looking result.

*(Copied as-is from `ingestion/dcs_codecs.py`, including a real tie-breaking bug we found and fixed while building this notebook — see the comment on `_UNMAPPED_DCS_TRY_ORDER` below.)*

In [5]:
# dcs number -> which decoder to use, for each source.
SMPP_DCS_TABLE: dict[int, str] = {
    0: "gsm7", 4: "gsm7", 241: "gsm7", 245: "gsm7",
    1: "ascii", 26: "ascii", 244: "ascii",
    8: "utf16", 24: "utf16", 25: "utf16",
    3: "latin1",
}

SS7_DCS_TABLE: dict[int, str] = {
    0: "gsm7", 12: "gsm7", 16: "gsm7", 17: "gsm7", 18: "gsm7",
    192: "gsm7", 241: "gsm7", 242: "gsm7", 245: "gsm7",
    8: "utf16", 24: "utf16", 25: "utf16",
    26: "ascii", 244: "ascii",
    # 1 and 3 deliberately absent - see SS7_AMBIGUOUS_DCS below.
}

# dcs numbers the spec genuinely lists under more than one meaning for SS7 -
# try each candidate, keep whichever decodes most legibly.
SS7_AMBIGUOUS_DCS: dict[int, tuple[str, ...]] = {
    1: ("gsm7", "ascii"),
    3: ("gsm7", "latin1"),
}

SMPP_CODEC_FUNCS = {
    "gsm7": decode_latin1,   # SMPP stores GSM-7-tagged content pre-unpacked,
                              # not as packed septets - see Step 1's note.
    "ascii": decode_ascii,
    "utf16": decode_utf16be,
    "latin1": decode_latin1,
}

SS7_CODEC_FUNCS = {
    "gsm7": decode_gsm7_packed,  # true packed septets
    "ascii": decode_ascii,
    "utf16": decode_utf16be,
    "latin1": decode_latin1,
}

_TABLES = {"SMPP": SMPP_DCS_TABLE, "SS7": SS7_DCS_TABLE}
_AMBIGUOUS = {"SMPP": {}, "SS7": SS7_AMBIGUOUS_DCS}
_FUNCS = {"SMPP": SMPP_CODEC_FUNCS, "SS7": SS7_CODEC_FUNCS}

# BUG WE FOUND AND FIXED: when a dcs number isn't in the table above at all,
# we try every decoder and keep the best-scoring one. But we found a real
# case where a correct, perfectly legible ASCII decode TIED (both scored a
# perfect 1.0) with an incorrect GSM-7 decode of the same bytes - GSM-7's
# table happens to map almost any byte to SOME printable character, so a
# wrong GSM-7 guess can still look 100% "printable" by accident. Ties used
# to go to whichever decoder was tried first, which happened to be the
# (wrong) GSM-7 one. Fix: try the simple, byte-preserving decoders first
# (ascii/latin1/utf16) - succeeding on one of those outright is a much
# stronger signal of being correct than GSM-7's bit-repacking happening to
# also look printable by chance.
_UNMAPPED_DCS_TRY_ORDER = ("ascii", "latin1", "utf16", "gsm7")


def _best_of(payload: bytes, codec_names, funcs: dict) -> tuple[str | None, str | None]:
    best_text, best_codec, best_score = None, None, -1.0
    for name in codec_names:
        text = funcs[name](payload)
        score = _printable_score(text)
        if score > best_score:
            best_text, best_codec, best_score = text, name, score
    return best_text, best_codec


def decode_by_dcs(payload: bytes, dcs: int | None, *, source: str) -> tuple[str | None, str | None]:
    """The main decision function. Give it the message bytes (header
    already removed - see Step 5), the dcs number, and which source it's
    from. Returns (decoded_text, which_decoder_was_used). which_decoder_was_
    used is tagged "(auto)" whenever it had to guess instead of using a
    direct table lookup, so you can always see how a given message was
    decoded."""
    if source not in _TABLES:
        raise ValueError(f"unknown source {source!r} - expected 'SMPP' or 'SS7'")
    if not payload:
        return None, None

    table, ambiguous, funcs = _TABLES[source], _AMBIGUOUS[source], _FUNCS[source]
    dcs_int = int(dcs) if dcs is not None else None

    if dcs_int in ambiguous:
        text, codec = _best_of(payload, ambiguous[dcs_int], funcs)
        codec_used = f"{codec}(auto)" if codec else None
    elif dcs_int in table:
        codec = table[dcs_int]
        text, codec_used = funcs[codec](payload), codec
    else:
        # dcs number not in our table at all - guess, best-scoring wins.
        candidates = [name for name in _UNMAPPED_DCS_TRY_ORDER if name in funcs]
        text, codec = _best_of(payload, candidates, funcs)
        codec_used = f"{codec}(auto)" if codec else None

    return _sanitize_for_storage(text), codec_used


print("decode_by_dcs ready. Example - SS7, dcs=0 (plain GSM-7 text), 'hello' packed:")
print(decode_by_dcs(bytes.fromhex("e8329bfd06"), 0, source="SS7"))

decode_by_dcs ready. Example - SS7, dcs=0 (plain GSM-7 text), 'hello' packed:
('hello', 'gsm7')


## Step 5 — removing the header bytes (UDH), when there is one

Some messages have a small header stuck on the front of the bytes — not part of what the user typed, but metadata like "this is part 2 of a 3-part message" or "this message is for application X, not for a person to read." If we don't strip this header off first, it shows up as garbled junk in front of (or instead of) the real text.

We detect this header by its *shape*, not by trusting a separate flag column: a real header is a specific byte-length followed by a sequence of (id, length, data) triples that exactly fill that length. Real message text essentially never happens to look like that by coincidence (checked against ~155,000 real messages: correctly found every real header, with about a 0.03% false-alarm rate).

*(Copied as-is from `ingestion/udh.py`.)*

In [6]:
_CONCAT_IEI_8BIT_REF = 0x00   # header field meaning: "multi-part message,
                              #  8-bit part reference" (3 bytes: ref, total
                              #  parts, this part's number)
_CONCAT_IEI_16BIT_REF = 0x08  # same, but 16-bit reference (4 bytes)
_PORT_IEI_8BIT = 0x04         # header field meaning: "addressed to
                              #  application port X" (8-bit port, 2 bytes)
_PORT_IEI_16BIT = 0x05        # same, but 16-bit port (4 bytes)


class UdhInfo(NamedTuple):
    present: bool
    header_len: int
    concat_ref: int | None            # multi-part message id, if this is one part of several
    concat_total_parts: int | None
    concat_part_num: int | None
    dest_port: int | None             # "addressed to application X" port number, if present


def parse_udh(raw: bytes) -> UdhInfo:
    """Looks at the START of the message bytes and checks: does this look
    like a well-formed header? If yes, also pulls out the multi-part info
    and/or the application port number, when present.

    BUG WE FOUND AND FIXED (via this notebook's own full-SS7-corpus run):
    the original version accepted ANY structurally well-formed header -
    a chain of (id, length, data) triples that exactly fills the declared
    length - regardless of whether it recognized any of the IE types in
    it. That's fine for single-byte text (GSM-7/ASCII/Latin-1), where a
    real message coincidentally forming that shape is ~impossible (~0.03%
    false-positive rate, measured on real SMPP data). But UTF-16BE content
    (dcs=8) has a MUCH higher coincidence rate: BMP characters put a 0x00
    (or small) byte every other position, so a chain of plausible-looking
    (iei, iel) pairs lines up far more easily. Two real examples found: a
    message starting with U+200E (LRM, bytes 0x20 0x0e) and a Chinese
    message, BOTH structurally "valid" headers that extracted NEITHER a
    recognized concat IE nor a recognized port IE - i.e. matched the shape
    but taught us nothing - and stripping that fake header left an ODD-
    length remainder, which decode_utf16be correctly refuses (needs an
    even byte count), turning perfectly legible real text into a spurious
    decode failure (text=None). Fix: only accept a header that actually
    yields a recognized IE (concat or port) - the only two kinds anything
    downstream reads anyway, so "structurally valid but recognizes
    nothing" is never useful and is exactly the shape this false positive
    takes. Real concat-only and port-only headers (SMPP multipart; SS7
    dcs=4 Application Port Addressing) still pass, since those DO
    populate one of the two."""
    if len(raw) < 2:
        return UdhInfo(False, 0, None, None, None, None)
    udhl = raw[0]  # first byte says how many header bytes follow
    if udhl == 0 or 1 + udhl > len(raw):
        return UdhInfo(False, 0, None, None, None, None)

    pos, end = 1, 1 + udhl
    concat_ref = concat_total = concat_part = dest_port = None
    while pos < end:
        if pos + 2 > end:
            return UdhInfo(False, 0, None, None, None, None)  # malformed
        iei, iel = raw[pos], raw[pos + 1]
        data_start = pos + 2
        if data_start + iel > end:
            return UdhInfo(False, 0, None, None, None, None)
        if iei == _CONCAT_IEI_8BIT_REF and iel == 3:
            concat_ref = raw[data_start]
            concat_total, concat_part = raw[data_start + 1], raw[data_start + 2]
        elif iei == _CONCAT_IEI_16BIT_REF and iel == 4:
            concat_ref = (raw[data_start] << 8) | raw[data_start + 1]
            concat_total, concat_part = raw[data_start + 2], raw[data_start + 3]
        elif iei == _PORT_IEI_8BIT and iel == 2:
            dest_port = raw[data_start]
        elif iei == _PORT_IEI_16BIT and iel == 4:
            dest_port = (raw[data_start] << 8) | raw[data_start + 1]
        pos = data_start + iel

    if pos != end:
        return UdhInfo(False, 0, None, None, None, None)  # didn't exactly fill the declared length
    if concat_ref is None and dest_port is None:
        # Structurally valid but recognized nothing - see the bug note
        # above. Not a real UDH match.
        return UdhInfo(False, 0, None, None, None, None)
    return UdhInfo(True, udhl, concat_ref, concat_total, concat_part, dest_port)


def strip_udh(payload: bytes, udh: UdhInfo) -> bytes:
    """Removes the header bytes, if there were any, leaving only the real message."""
    if not udh.present or len(payload) == 0:
        return payload
    return payload[1 + udh.header_len:]


print("UDH parser ready.")

UDH parser ready.


## Step 6 — putting it together, one function per data source

These are the two functions actually used by the real pipeline (`ingestion/smpp.py::_decode_row` and `ingestion/ss7.py::_decode_row`), copied here under clearer names. Each one: strips the header if present, corrects a source-specific quirk in how `dcs` is stored, then calls `decode_by_dcs` from Step 4.

**The bug we found and fixed, explained simply:** one particular `dcs` value on our SS7 source (`4`) doesn't mean "text" at all — per the telecom spec it means "raw binary data for some application, not a message a person typed." Before the fix, we didn't know this, so we tried to decode it as text anyway and got confident-looking gibberish (garbled symbols that still "look like" a valid decode). We checked real examples of this and found:
- About a third of the time, this binary data DOES have a header (Step 5) saying which application it's for — and once you remove that header, what's left really is a short piece of legible text (looks like an ID code or token).
- The rest of the time, there's no header and no way to know what the binary data means — so now we correctly say "we don't know" (`text = None`) instead of guessing.

In [7]:
# dcs value(s) that mean "binary data for an application", not text - SS7
# only. (Our SMPP source uses dcs=4 to mean ordinary GSM-7 text instead -
# confirmed separately - this is exactly the "same number, different source,
# different meaning" trap mentioned in Step 4.)
SS7_BINARY_DATA_CLASS_DCS = {4}


def smpp_decode_row(content_hex, dcs) -> dict:
    """SMPP source: always has a header to check for (used for multi-part
    messages). dcs is stored as a signed number upstream and needs
    `% 256` to become the real, correct unsigned value."""
    empty = {"text": None, "concat_ref": None, "concat_total_parts": None,
             "concat_part_num": None, "udh_dest_port": None}
    if not isinstance(content_hex, str) or not content_hex:
        return empty
    try:
        raw_bytes = bytes.fromhex(content_hex)
    except ValueError:
        return empty

    udh = parse_udh(raw_bytes)
    payload = strip_udh(raw_bytes, udh)

    dcs_byte = int(dcs) % 256 if pd.notna(dcs) else None
    text, _codec_used = decode_by_dcs(payload, dcs_byte, source="SMPP")

    return {
        "text": text,
        "concat_ref": udh.concat_ref,
        "concat_total_parts": udh.concat_total_parts,
        "concat_part_num": udh.concat_part_num,
        "udh_dest_port": udh.dest_port,
    }


def ss7_decode_row(content_hex, dcs) -> dict:
    """SS7 source: multi-part info comes from separate columns elsewhere,
    not from a header - but a header can still show up for OTHER reasons
    (the binary/application-data case below), so we still check for one.
    dcs is already the correct unsigned number here, no correction needed."""
    empty = {"text": None, "content_is_binary": False, "udh_dest_port": None}
    if not isinstance(content_hex, str) or not content_hex:
        return empty
    try:
        raw_bytes = bytes.fromhex(content_hex)
    except ValueError:
        return empty

    udh = parse_udh(raw_bytes)
    payload = strip_udh(raw_bytes, udh)
    dcs_int = int(dcs) if pd.notna(dcs) else None

    if dcs_int in SS7_BINARY_DATA_CLASS_DCS and not udh.present:
        # Binary data, no header to explain it - don't guess a text decode.
        return {"text": None, "content_is_binary": True, "udh_dest_port": udh.dest_port}

    text, _codec_used = decode_by_dcs(payload, dcs_int, source="SS7")
    return {"text": text, "content_is_binary": False, "udh_dest_port": udh.dest_port}


print("Both source-specific decoders ready. All the building blocks above are now in place.")

Both source-specific decoders ready. All the building blocks above are now in place.


## Now: prove it works, on real messages

Everything above is the actual logic. From here on, we run it against real sample data and check the results.

### Load some real messages

Grabs a few thousand real rows from our raw data files, for each source.

In [8]:
N_FILES = 3
SAMPLE_PER_FILE = 2000

smpp_files = sorted((REPO_ROOT / "data/raw/SMPP").glob("*/*.csv"))[:N_FILES]
ss7_files = sorted((REPO_ROOT / "data/raw/SS7").glob("*/*.csv"))[:N_FILES]

smpp_raw = pd.concat(
    [pd.read_csv(f, usecols=["content", "dcs", "decoded_content"], dtype=str, nrows=SAMPLE_PER_FILE) for f in smpp_files],
    ignore_index=True,
).dropna(subset=["content"])

ss7_raw = pd.concat(
    [pd.read_csv(f, usecols=["content", "dcs", "decoded_content"], dtype=str, nrows=SAMPLE_PER_FILE) for f in ss7_files],
    ignore_index=True,
).dropna(subset=["content"])

print(f"SMPP: {len(smpp_raw)} sample rows from {len(smpp_files)} file(s)")
print(f"SS7:  {len(ss7_raw)} sample rows from {len(ss7_files)} file(s)")
smpp_raw.head(3)

SMPP: 2984 sample rows from 3 file(s)


SS7:  1348 sample rows from 3 file(s)


,dcs,content,decoded_content
1,0,0500034902026c69636b20746865206c696e6b20696e20746865206d6573736167652e,é@¥I$$lick the link in the message.
2,-15,524d3020536574656c2e20444f204e4f542053484152452054484953204f54502e20596f7572...,RM0 Setel. DO NOT SHARE THIS OTP. Your OTP is 480790. Valid for 5 mins. Cont...
4,-15,524d3020434f5741593a204865726520636f6d657320616e6f74686572206d6f6e7468212041...,RM0 COWAY: Here comes another month! AUG-26 bill of RM94.00 for your invoice...


### Run the decoders on every sample row

`codec_used` records which decoder actually got used for each row (and whether it was a direct table lookup or a best-guess), so we can see the breakdown per `dcs` value below.

In [9]:
def decode_with_codec(content_hex, dcs, *, source):
    """Runs the real per-source decoder (smpp_decode_row / ss7_decode_row)
    for the actual text/flags, and separately replays parse_udh -> strip_udh
    -> decode_by_dcs just to also capture which codec was used, for display
    (same steps, same inputs - just also keeping a value the wrapper
    functions above don't return)."""
    decode_row_fn = smpp_decode_row if source == "SMPP" else ss7_decode_row
    result = decode_row_fn(content_hex, dcs)

    codec_used = None
    if isinstance(content_hex, str) and content_hex:
        try:
            raw = bytes.fromhex(content_hex)
        except ValueError:
            raw = b""
        if raw:
            u = parse_udh(raw)
            payload = strip_udh(raw, u)
            dcs_int = None if pd.isna(dcs) else (int(dcs) % 256 if source == "SMPP" else int(dcs))
            if not (source == "SS7" and dcs_int in SS7_BINARY_DATA_CLASS_DCS and not u.present):
                _, codec_used = decode_by_dcs(payload, dcs_int, source=source)

    return result.get("text"), codec_used, result.get("content_is_binary", False), result.get("udh_dest_port")

smpp_raw[["text", "codec_used", "content_is_binary", "udh_dest_port"]] = smpp_raw.apply(
    lambda r: pd.Series(decode_with_codec(r["content"], r["dcs"], source="SMPP")), axis=1
)
ss7_raw[["text", "codec_used", "content_is_binary", "udh_dest_port"]] = ss7_raw.apply(
    lambda r: pd.Series(decode_with_codec(r["content"], r["dcs"], source="SS7")), axis=1
)

smpp_raw["printable_score"] = smpp_raw["text"].apply(_printable_score)
ss7_raw["printable_score"] = ss7_raw["text"].apply(_printable_score)

ss7_raw[["dcs", "content", "text", "codec_used", "content_is_binary"]].head(10)

,dcs,content,text,codec_used,content_is_binary
0,0,d3f2b82e4fd3f3a0797e4e2fb741f437485e6ea7dd6450fe5dd73514d474bbac93c164b6170c...,Security system to remind you: Time:2026/08/02 00:02:22 DevName:SEN303-SBNLI...,gsm7,False
2,0,c4e23408b226a7d420088a04318bd66223485d0d9bc7694f048be56a30182c569b891ac4e234...,"DESA VISTA PH LEVE AT+CMGS=""0195000153"" DESA VISTA PH LEVEL HH OK",gsm7,False
9,0,4434481d6687dda06b9a0d9ae7cf,Dh jalan Wil syg,gsm7,False
14,4,0200061e3a130905f22135ee33b3e03f3a8a662708666451073a980610415071767058f43a08...,None,None,True
26,4,06050404d200002854325a4b614d6a4b6866347729,(T2ZKaMjKhf4w),ascii(auto),False
35,4,020011493a130705f22107ed28dfffff3a8a965103140349003a980610411122030850f53a08...,None,None,True
40,0,cbb23b0c0f83d6e13a489c769febee33681e769fc374,Kenapa kau bingung sangat,gsm7,False
41,0,d3f2b82e4fd3f3a0797e4e2fb741f437485e6ea7dd6450fe5dd73514d474bbac93c164b6170c...,Security system to remind you: Time:2026/08/01 23:59:38 DevName:LPG096T-RIAJ...,gsm7,False
43,0,cf3508,Ok,gsm7,False
45,0,2858f34a1e8bcb4e5c34ae4e01,(0MWdcbeN8Qqj),gsm7,False


### Summary table — the main thing to hand off

For each `dcs` value actually seen in the sample: how many messages, which decoder was used, and how legible the results look on average (`mean_printable_score`, from Step 3). A `dcs` value where that score is unexpectedly low for real messages is the signal to go investigate — like the SS7 `dcs=4` case documented below.

In [10]:
def summarize(df, source):
    g = df.groupby(["dcs", "codec_used"], dropna=False).agg(
        n_rows=("text", "size"),
        mean_printable_score=("printable_score", "mean"),
        example_text=("text", lambda s: next((t for t in s if t), None)),
    ).reset_index()
    g.insert(0, "source", source)
    return g.sort_values(["dcs", "n_rows"], ascending=[True, False])

summary = pd.concat([summarize(smpp_raw, "SMPP"), summarize(ss7_raw, "SS7")], ignore_index=True)
summary

,source,dcs,codec_used,n_rows,mean_printable_score,example_text
0,SMPP,-15,gsm7,2754,0.999132,RM0 Setel. DO NOT SHARE THIS OTP. Your OTP is 480790. Valid for 5 mins. Cont...
1,SMPP,0,gsm7,143,0.997512,lick the link in the message.
2,SMPP,25,utf16,34,0.997693,"RM0 Ryt Bank: 检测到新设备绑定。从 1 Aug 2026, 11:59 PM (GMT+8) 起已启动 12 小时冷静期。"
3,SMPP,8,utf16,53,0.987421,楤㨰ㄴ㔹㠴ㄲ㈠獵戺〰〠摬癲携〰〠獵扭楴⁤慴攺㈶〸〱〰〰⁤潮攠摡瑥㨲㘰㠰㈰〰〠獴慴㩅塐䥒䕄⁥牲㨰〸⁔數琺
4,SS7,0,gsm7,914,0.999944,Security system to remind you: Time:2026/08/02 00:02:22 DevName:SEN303-SBNLI...
5,SS7,241,gsm7,3,1.000000,Alarm @ RPKB JLN SG BESI 01:00:02 02-08-2026 PUMP 02 START
6,SS7,3,gsm7(auto),1,1.000000,?n001801025A05156F2000000000150000000194928100000000000015082005062600000000...
7,SS7,4,NaN,236,0.000000,None
8,SS7,4,ascii(auto),128,1.000000,(T2ZKaMjKhf4w)
9,SS7,8,utf16,66,1.000000,"n kita, dpt jumpa.kita akn abik apa brg yg ada Dr ag.ag dealead smu"


### Sanity check against the raw data's own pre-decoded column

The raw files already come with a `decoded_content` column decoded by an upstream system, before ours. Comparing against it is a useful sanity check, with one caveat already known and expected: our SMPP decoder deliberately does its OWN decoding rather than trusting that column, because that column was found to leave header bytes (Step 5) stuck onto the front of the text on that source — a real bug in the upstream column, not in ours. So SMPP rows are EXPECTED to disagree there; SS7 rows are expected to closely agree.

(Whitespace is normalized before comparing, since our sanitizing step in Step 3 turns line breaks into spaces on purpose — that's not a real disagreement.)

In [11]:
def normalize_ws(s):
    return re.sub(r"\s+", " ", s).strip() if isinstance(s, str) else s

def agreement_rate(df, label):
    both = df.dropna(subset=["text", "decoded_content"])
    if len(both) == 0:
        print(f"{label}: no rows with both text and decoded_content present")
        return
    match = both["text"].apply(normalize_ws) == both["decoded_content"].apply(normalize_ws)
    print(f"{label}: {match.mean():.1%} agreement over {len(both)} rows")

agreement_rate(smpp_raw, "SMPP (expected to DISAGREE where the upstream column leaks header bytes)")
agreement_rate(ss7_raw, "SS7 (expected to closely AGREE)")

SMPP (expected to DISAGREE where the upstream column leaks header bytes): 74.7% agreement over 2984 rows
SS7 (expected to closely AGREE): 84.0% agreement over 1112 rows


### The bug we found: SS7 `dcs=4` messages, before vs after the fix

Explained above in Step 6. Short version: `dcs=4` on our SS7 source means "binary data for an application," not text. This check confirms it's a real, sizeable issue (not a rare edge case) and shows the fix's actual effect.

In [12]:
dcs4 = ss7_raw[ss7_raw["dcs"] == "4"].copy()
dcs0 = ss7_raw[ss7_raw["dcs"] == "0"].copy()

def ascii_letter_ratio(text):
    """How much of this text is plain letters/spaces? Real English messages
    score high here. Unlike printable_score (Step 3), this isn't fooled by
    a wrong decode that happens to produce 'exotic but technically
    printable' characters (accented letters, Greek letters, etc.) - which
    is exactly what tricked printable_score in this case."""
    if not text:
        return 0.0
    letters = sum(1 for ch in text if ch.isascii() and (ch.isalpha() or ch == " "))
    return letters / len(text)

dcs4["ascii_letter_ratio"] = dcs4["text"].apply(ascii_letter_ratio)
dcs0["ascii_letter_ratio"] = dcs0["text"].apply(ascii_letter_ratio)

recovered = dcs4[~dcs4["content_is_binary"]]
still_binary = dcs4[dcs4["content_is_binary"]]

print(f"{len(dcs4)} total dcs=4 messages in this sample")
print(f"  -> {len(recovered)} had a header explaining what app they're for - header removed, "
      f"decoded normally (average letter-likeness: {recovered['ascii_letter_ratio'].mean():.2f}, "
      f"vs {dcs0['ascii_letter_ratio'].mean():.2f} for known-good dcs=0 English text)")
print(f"  -> {len(still_binary)} had no such header - correctly left as 'unknown' (text=None) "
      "instead of a fake guess")
print()
print("Example of a recovered message (header removed, then decoded):")
recovered[["content", "udh_dest_port", "text"]].head(3)

364 total dcs=4 messages in this sample
  -> 128 had a header explaining what app they're for - header removed, decoded normally (average letter-likeness: 0.72, vs 0.74 for known-good dcs=0 English text)
  -> 236 had no such header - correctly left as 'unknown' (text=None) instead of a fake guess

Example of a recovered message (header removed, then decoded):


,content,udh_dest_port,text
26,06050404d200002854325a4b614d6a4b6866347729,1234.0,(T2ZKaMjKhf4w)
56,06050404d2000028727a7735304d70725a48684829,1234.0,(rzw50MprZHhH)
57,06050404d2000028645a43525346314c6d73646b29,1234.0,(dZCRSF1Lmsdk)


## Save the summary table

Writes the per-(source, dcs) summary to a CSV next to this notebook, so the numbers can be shared without anyone needing to re-run anything.

In [13]:
out_path = REPO_ROOT / "notebooks" / "decode_verification_summary.csv"
summary.to_csv(out_path, index=False)
print("wrote", out_path)

wrote C:\Users\IshitaGodani\Documents\projects\spam-detection-prototype\notebooks\decode_verification_summary.csv


## Full-corpus check — how much of ALL our data decodes correctly?

Everything above only sampled `N_FILES=3` files per source, to keep the notebook fast to read
through. This section runs the same decoders over **every** SMPP and SS7 raw file we have, to get
a real coverage number instead of a sample estimate.

**What "decoded correctly" means here**, in this codebase's own terms — there's no ground-truth
label to check against, so we use the same two signals the pipeline itself relies on:
- **not empty/failed**: `text` came back non-null (i.e. `decode_by_dcs` didn't give up — see Step 4)
  and, for SS7, it wasn't flagged `content_is_binary` (real binary payload, not text at all —
  Step 6). This is exactly what the pipeline's own `text_decode_failed` canonical column
  (`common/schemas.py`) tracks for SMPP; we compute the equivalent for both sources here.
- **legible**: among the rows that got *some* text back, `_printable_score` (Step 3) is high — a
  low score on a non-null decode usually means we picked the wrong codec (see the `dcs=4`
  gsm7-vs-ascii tie-break bug fixed in Step 6), not that the message is genuinely unreadable.

Processed one file at a time (not concatenated into one giant DataFrame) so this scales to however
many files exist without needing all of them in memory at once — only small per-file/per-source
running totals are kept.

In [14]:
LEGIBLE_PRINTABLE_SCORE_THRESHOLD = 0.9  # same spirit as FAISS_NEAR_DUP_THRESHOLD in
# config/settings.py - a starting point, not calibrated against a labelled set.


def check_decode_coverage(source: str, files, *, printable_threshold: float = LEGIBLE_PRINTABLE_SCORE_THRESHOLD):
    """Runs the real smpp_decode_row/ss7_decode_row decoder (Step 6) over
    every row of every file given, one file at a time, and returns a dict
    of corpus-wide totals: how many rows decoded at all (non-null text,
    and for SS7 not content_is_binary), and of those, how many look
    legible (printable_score >= printable_threshold).

    Deliberately does NOT hold every row's decoded text in memory - only
    running counts survive past each file, so this works the same way
    whether given 3 files or all of them (same reasoning as
    FAISS_CHUNK_SIZE bounding features/faiss_index.py's memory - see
    config/settings.py)."""
    decode_row_fn = smpp_decode_row if source == "SMPP" else ss7_decode_row

    total_rows = 0
    empty_content = 0       # content missing/unparseable before decoding even starts
    decode_failed = 0       # content present but decode_by_dcs gave up (text is None),
                             # or SS7 content_is_binary=True (real binary, not text)
    decoded_legible = 0     # text non-null AND printable_score >= threshold
    decoded_illegible = 0   # text non-null but printable_score < threshold (wrong codec guess)
    per_file_rows = []

    for f in files:
        df = pd.read_csv(f, usecols=["content", "dcs"], dtype=str)
        total_rows += len(df)

        missing = df["content"].isna() | (df["content"] == "")
        empty_content += missing.sum()
        present = df[~missing]

        results = present.apply(lambda r: decode_row_fn(r["content"], r["dcs"]), axis=1)
        text = results.apply(lambda r: r.get("text"))
        is_binary = results.apply(lambda r: r.get("content_is_binary", False)) if source == "SS7" else False

        failed_mask = text.isna() | (is_binary if source == "SS7" else False)
        decode_failed += failed_mask.sum()

        ok_text = text[~failed_mask]
        scores = ok_text.apply(_printable_score)
        legible_mask = scores >= printable_threshold
        decoded_legible += legible_mask.sum()
        decoded_illegible += (~legible_mask).sum()

        per_file_rows.append({"file": f.name, "rows": len(df)})

    decoded_ok = decoded_legible + decoded_illegible  # non-null text, regardless of legibility
    return {
        "source": source,
        "n_files": len(files),
        "total_rows": total_rows,
        "empty_content": empty_content,
        "decode_failed": decode_failed,
        "decoded_legible": decoded_legible,
        "decoded_illegible": decoded_illegible,
        "decode_success_rate": decoded_ok / total_rows if total_rows else float("nan"),
        "legible_rate_of_total": decoded_legible / total_rows if total_rows else float("nan"),
        "legible_rate_of_decoded": decoded_legible / decoded_ok if decoded_ok else float("nan"),
        "per_file": pd.DataFrame(per_file_rows),
    }


print("check_decode_coverage() ready.")

check_decode_coverage() ready.


In [15]:
all_smpp_files = sorted((REPO_ROOT / "data/raw/SMPP").glob("*/*.csv"))
all_ss7_files = sorted((REPO_ROOT / "data/raw/SS7").glob("*/*.csv"))
print(f"Found {len(all_smpp_files)} SMPP file(s), {len(all_ss7_files)} SS7 file(s) - running full check...")

smpp_coverage = check_decode_coverage("SMPP", all_smpp_files)
ss7_coverage = check_decode_coverage("SS7", all_ss7_files)

coverage_report = pd.DataFrame([
    {k: v for k, v in smpp_coverage.items() if k != "per_file"},
    {k: v for k, v in ss7_coverage.items() if k != "per_file"},
])
coverage_report

Found 48 SMPP file(s), 48 SS7 file(s) - running full check...


,source,n_files,total_rows,empty_content,decode_failed,decoded_legible,decoded_illegible,decode_success_rate,legible_rate_of_total,legible_rate_of_decoded
0,SMPP,48,14101168,7053262,168,7038862,8876,0.499798,0.499169,0.998741
1,SS7,48,14685436,11269128,472730,2942338,1240,0.200442,0.200358,0.999579


In [16]:
coverage_out_path = REPO_ROOT / "notebooks" / "decode_coverage_full_corpus.csv"
coverage_report.to_csv(coverage_out_path, index=False)
print("wrote", coverage_out_path)

for r in (smpp_coverage, ss7_coverage):
    print(f"\n{r['source']}: {r['total_rows']:,} rows across {r['n_files']} file(s)")
    print(f"  decode_success_rate (non-null text): {r['decode_success_rate']:.2%}")
    print(f"  legible_rate_of_total (score >= {LEGIBLE_PRINTABLE_SCORE_THRESHOLD}): {r['legible_rate_of_total']:.2%}")
    print(f"  legible_rate_of_decoded: {r['legible_rate_of_decoded']:.2%}")
    print(f"  empty_content: {r['empty_content']:,}, decode_failed: {r['decode_failed']:,}, "
          f"decoded_illegible: {r['decoded_illegible']:,}")

wrote C:\Users\IshitaGodani\Documents\projects\spam-detection-prototype\notebooks\decode_coverage_full_corpus.csv

SMPP: 14,101,168 rows across 48 file(s)
  decode_success_rate (non-null text): 49.98%
  legible_rate_of_total (score >= 0.9): 49.92%
  legible_rate_of_decoded: 99.87%
  empty_content: 7,053,262, decode_failed: 168, decoded_illegible: 8,876

SS7: 14,685,436 rows across 48 file(s)
  decode_success_rate (non-null text): 20.04%
  legible_rate_of_total (score >= 0.9): 20.04%
  legible_rate_of_decoded: 99.96%
  empty_content: 11,269,128, decode_failed: 472,730, decoded_illegible: 1,240


In [17]:
all_ss7_files = sorted((REPO_ROOT / "data/raw/SS7").glob("*/*.csv"))

# Same row filter the real pipeline applies (ingestion/ss7.py::clean()) -
# only message_type 2 (MT_request) and 3 (MO) ever carry real content;
# 1/4/5 (SRI_request/SRI_response/MT_response) are always contentless
# protocol-flow rows and would otherwise inflate "empty content" with rows
# that were never going to have a message in the first place.
SS7_CONTENT_MESSAGE_TYPES = {2, 3}


def ss7_decode_with_codec(content_hex, dcs):
    """Same logic as ss7_decode_row (Step 6), but decodes ONCE and also
    returns which codec was used - ss7_decode_row/decode_with_codec above
    decoded twice per row (once for the result, once replayed just to
    recover codec_used), which is wasteful over millions of rows."""
    if not isinstance(content_hex, str) or not content_hex:
        return None, None, False
    try:
        raw = bytes.fromhex(content_hex)
    except ValueError:
        return None, None, False

    u = parse_udh(raw)
    payload = strip_udh(raw, u)
    dcs_int = int(dcs) if pd.notna(dcs) else None

    if dcs_int in SS7_BINARY_DATA_CLASS_DCS and not u.present:
        return None, None, True

    text, codec_used = decode_by_dcs(payload, dcs_int, source="SS7")
    return text, codec_used, False


n_readable = n_binary = n_empty = n_failed = n_suspicious = n_unknown_codec = n_total = 0
examples = {"suspicious": [], "failed": [], "unknown_codec": []}
MAX_EXAMPLES = 2

for f in all_ss7_files:
    df = pd.read_csv(f, usecols=["content", "dcs", "message_type"], dtype=str)
    df = df[df["message_type"].astype(float).astype("Int64").isin(SS7_CONTENT_MESSAGE_TYPES)]
    n_total += len(df)

    missing = df["content"].isna() | (df["content"] == "")
    n_empty += missing.sum()
    df = df[~missing]

    decoded = df.apply(lambda r: ss7_decode_with_codec(r["content"], r["dcs"]), axis=1)
    text = decoded.apply(lambda t: t[0])
    codec_used = decoded.apply(lambda t: t[1])
    is_binary = decoded.apply(lambda t: t[2])
    scores = text.apply(_printable_score)

    n_binary += is_binary.sum()

    failed_mask = ~is_binary & text.isna()
    n_failed += failed_mask.sum()
    suspicious_mask = ~is_binary & text.notna() & (scores < LEGIBLE_PRINTABLE_SCORE_THRESHOLD)
    n_suspicious += suspicious_mask.sum()
    n_readable += (~is_binary & text.notna() & (scores >= LEGIBLE_PRINTABLE_SCORE_THRESHOLD)).sum()

    # "unknown codec": dcs wasn't in our lookup table, had to guess (Step 4) -
    # counted separately, can overlap with any of the categories above.
    unknown_codec_mask = codec_used.fillna("").str.endswith("(auto)")
    n_unknown_codec += unknown_codec_mask.sum()

    for name, mask in (("failed", failed_mask), ("suspicious", suspicious_mask), ("unknown_codec", unknown_codec_mask)):
        if len(examples[name]) < MAX_EXAMPLES:
            for idx in df[mask].index[:MAX_EXAMPLES - len(examples[name])]:
                examples[name].append({
                    "content": df.loc[idx, "content"], "dcs": df.loc[idx, "dcs"],
                    "text": text.loc[idx], "codec_used": codec_used.loc[idx],
                })


def _row(label, n):
    print(f"{label:<20}: {n:>10,} ({n / n_total:.2%})")

print("=" * 65)
print("DECODING CATEGORIES (SS7, message_type in {2, 3} only - matches ingestion/ss7.py::clean())")
print("=" * 65)
_row("Readable text", n_readable)
_row("Binary content", n_binary)
_row("Empty content", n_empty)
_row("Decode failed", n_failed)
_row("Suspicious text", n_suspicious)
_row("Unknown codec", n_unknown_codec)

for name, label in (("suspicious", "SUSPICIOUS TEXT"), ("failed", "DECODE FAILED"), ("unknown_codec", "UNKNOWN CODEC")):
    print(f"\n--- {label} examples ---")
    if not examples[name]:
        print("  (none found)")
    for ex in examples[name]:
        print(f"  dcs={ex['dcs']}  codec_used={ex['codec_used']}  text={ex['text']!r}")
        print(f"    content={ex['content']}")

DECODING CATEGORIES (SS7, message_type in {2, 3} only - matches ingestion/ss7.py::clean())
Readable text       :  2,942,338 (86.10%)
Binary content      :    472,719 (13.83%)
Empty content       :      1,117 (0.03%)
Decode failed       :         11 (0.00%)
Suspicious text     :      1,240 (0.04%)
Unknown codec       :    141,510 (4.14%)

--- SUSPICIOUS TEXT examples ---
  dcs=8  codec_used=utf16  text='RITICAL\u2000at\u20002026-08-02\u200000:50:49\u2000MBB\u2000NTT(NTTDC))\u2000\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
    content=005200490054004900430041004c20000061007420000032003000320036002d00300038002d00300032200000300030003a00350030003a003400392000004d004200422000004e005400540028004e0054005400440043002900292000000000000000000000000000000000000000000000000000
  dcs=8  codec_used=utf16  text='\u200001:06:23\u2000WINTEL)\u2000\x00\x00\x00\x00\x00\x00'
    content=200000300031003a00300036003a00320033200000570049004e00540045004c00292000000000000000000000000000

--- DECODE FAILE

## Scratch cell — decode one message by hand

Paste a `content` hex string and a `dcs` value below (from a row you're looking at, e.g. in the
categories above or from an external tool) and run this cell to see exactly how our decoder
handles it: UDH detection, which codec got picked, the resulting text, and its printable score.

In [45]:
# --- Edit these three lines, then run the cell ---
content_hex = "621173b057284e004e2a4eba9a7e8f66ff0c57288def4e0a603b662f4e0076f454ed7740ff0c771f7684ff0c4f6065b94fbf62534e2a75358bdd7ed96211ff0c5b8961706211597d5417ff1f6211771f76845f887d2f5f887d2fff0c6211771f768430024e0d77e59053600e4e48529e3002"
dcs = 8
source = "SS7"  # "SS7" or "SMPP"
# ---------------------------------------------------

raw = bytes.fromhex(content_hex)
udh = parse_udh(raw)
payload = strip_udh(raw, udh)
dcs_int = int(dcs) % 256 if source == "SMPP" else int(dcs)
text, codec_used = decode_by_dcs(payload, dcs_int, source=source)

print(f"raw bytes         : {raw!r}")
print(f"udh.present        : {udh.present}")
if udh.present:
    print(f"  header_len       : {udh.header_len}")
    print(f"  concat_ref/total/part: {udh.concat_ref}/{udh.concat_total_parts}/{udh.concat_part_num}")
    print(f"  dest_port        : {udh.dest_port}")
print(f"payload (post-UDH) : {payload!r}")
print(f"codec_used         : {codec_used}")
print(f"text               : {text!r}")
print(f"printable_score    : {_printable_score(text):.3f}")

raw bytes         : b'b\x11s\xb0W(N\x00N*N\xba\x9a~\x8ff\xff\x0cW(\x8d\xefN\n`;f/N\x00v\xf4T\xedw@\xff\x0cw\x1fv\x84\xff\x0cO`e\xb9O\xbfbSN*u5\x8b\xdd~\xd9b\x11\xff\x0c[\x89apb\x11Y}T\x17\xff\x1fb\x11w\x1fv\x84_\x88}/_\x88}/\xff\x0cb\x11w\x1fv\x840\x02N\rw\xe5\x90S`\x0eNHR\x9e0\x02'
udh.present        : False
payload (post-UDH) : b'b\x11s\xb0W(N\x00N*N\xba\x9a~\x8ff\xff\x0cW(\x8d\xefN\n`;f/N\x00v\xf4T\xedw@\xff\x0cw\x1fv\x84\xff\x0cO`e\xb9O\xbfbSN*u5\x8b\xdd~\xd9b\x11\xff\x0c[\x89apb\x11Y}T\x17\xff\x1fb\x11w\x1fv\x84_\x88}/_\x88}/\xff\x0cb\x11w\x1fv\x840\x02N\rw\xe5\x90S`\x0eNHR\x9e0\x02'
codec_used         : utf16
text               : '我现在一个人驾车，在路上总是一直哭着，真的，你方便打个电话给我，安慰我好吗？我真的很累很累，我真的。不知道怎么办。'
printable_score    : 1.000
